# CMOS RGB image sensor in BeamZ

This notebook is a BeamZ implementation of Flexcompute's **[Tidy3D CMOS RGB image sensor notebook](https://www.flexcompute.com/tidy3d/examples/notebooks/CMOSRGBSensor/)**, which is explicitly the reference that we aim to replicate. It follows the reference's 3D RGGB Bayer cell: four silica microlenses, absorptive color filters, an aluminum shield and interconnects, a silicon-nitride antireflection layer, a silicon detector, normal-incidence illumination, field views, and per-pixel optical efficiency.

Two BeamZ-specific substitutions are made transparently. Zero-phase periodic boundaries reproduce the reference's repeated cell in $x$ and $y$, while absorbing layers truncate $z$. BeamZ does not yet expose causal dispersive media, so the broadband reference calculation is expressed as independent narrowband simulations. At each wavelength, tabulated/analytic $n(\lambda),k(\lambda)$ values are converted to nondispersive relative permittivity and electrical conductivity. This is appropriate for a frequency sweep, but each wavelength is a separate FDTD run.

The full notebook uses 31 wavelengths from 400 to 700 nm. `BEAMZ_DOCS_TEST=1` selects one deliberately coarse green-light case for automated smoke testing.

## References carried over from the reference notebook

- [Tidy3D CMOS RGB image sensor notebook](https://www.flexcompute.com/tidy3d/examples/notebooks/CMOSRGBSensor/) — our primary reference.
- [Tidy3D material library](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/material_library.html) and [FastDispersionFitter](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.plugins.dispersion.FastDispersionFitter.html) — the material sources and fitting workflow used by the reference.
- [Modeling dispersive material in FDTD](https://www.flexcompute.com/fdtd101/Lecture-5-Modeling-dispersive-material-in-FDTD) — background cited by the reference for causal material fitting.
- Further color-filter and color-routing approaches cited there: [pixelated color-filter arrays](https://www.mdpi.com/1424-8220/19/7/1536), [plasmonic color filters](https://opg.optica.org/oe/fulltext.cfm?uri=oe-16-25-20457&id=175035), [CMOS-compatible subwavelength color filters](https://ieeexplore.ieee.org/document/5427134), and [metasurface-based color routing](https://www.nature.com/articles/s41467-022-31019-7).

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
try:
    from IPython.display import display
except ImportError:
    display = print

import beamz as bz

test_mode = os.environ.get("BEAMZ_DOCS_TEST") == "1"
# Periodic boundaries currently execute through BeamZ's JAX kernels.
execution_backend = "jax"
um = bz.um


## 1. Bayer-cell geometry

The following dimensions reproduce the unoptimized values in the reference notebook. The 3 µm × 3 µm cell contains four 1.5 µm-wide pixels in the arrangement red (lower left), blue (upper right), and green (upper left and lower right).

In [ ]:
Lx = Ly = 3.0 * um
d_absorber_space = 1.5 * um

pixel_width = Lx / 2
lens_radius = 1.0 * um
lens_height = (
    1 - np.cos(np.arcsin(pixel_width / (2 * lens_radius)))
) * lens_radius

d_lens_to_filter = 0.2 * um
t_filter = 1.0 * um
d_filter_to_shield = 0.5 * um
t_shield = 0.10 * um
shield_hole_radius = 0.95 * pixel_width
d_shield_to_connect = 0.5 * um
t_connect = 0.10 * um
connect_width = 0.15 * um
connect_gap = 0.2 * um
connect_distance = 1.8 * pixel_width
d_connect_to_ar = 0.5 * um
t_silicon = 2.0 * um

lam_blue = 0.45 * um
lam_green = 0.55 * um
lam_red = 0.65 * um
n_sin = 2.05
t_ar = lam_green / (4 * n_sin)

Lz = (
    d_absorber_space + lens_height + d_lens_to_filter + t_filter
    + d_filter_to_shield + t_shield + d_shield_to_connect
    + 2 * t_connect + connect_gap + d_connect_to_ar + t_ar + t_silicon
)
print(f"lens cap height = {lens_height / um:.3f} µm")
print(f"SiN quarter-wave thickness = {t_ar / um:.3f} µm")
print(f"simulation domain = {Lx / um:.2f} × {Ly / um:.2f} × {Lz / um:.2f} µm³")


## 2. Narrowband material models

For a single angular frequency $\omega$, an optical index $n+ik$ is represented with

$$\epsilon_r=n^2-k^2, \qquad \sigma=2nk\omega\epsilon_0.$$

The artificial RGB filters reproduce the reference's intended step-like extinction: low $k$ in the channel passband and high $k$ elsewhere, including its baseline $k=10^{-2}$. The silicon interpolation is a compact visible-band approximation. Silica and SiN are held constant, while aluminum is represented by a conductive metal. These models are illustrative rather than replacements for measured process data or the reference's passive pole-residue fits.

In [ ]:
def material_from_nk(n, k, wavelength):
    omega = 2 * np.pi * bz.LIGHT_SPEED / wavelength
    epsilon_r = n**2 - k**2
    conductivity = 2 * n * k * omega * bz.EPS_0
    return bz.Material(permittivity=epsilon_r, conductivity=conductivity)


def logistic(value):
    return 1.0 / (1.0 + np.exp(-value))


def filter_nk(wavelength, channel):
    lam_um = wavelength / um
    edge = 0.012
    if channel == "blue":
        rejection = logistic((lam_um - 0.50) / edge)
    elif channel == "red":
        rejection = logistic((0.60 - lam_um) / edge)
    elif channel == "green":
        rejection = logistic((0.49 - lam_um) / edge) + logistic((lam_um - 0.61) / edge)
    else:
        raise ValueError(f"unknown filter channel: {channel}")
    return 1.60, 0.01 + 0.90 * rejection


def silicon_nk(wavelength):
    wavelength_um = wavelength / um
    knots = np.array([0.40, 0.45, 0.55, 0.65, 0.70])
    n_values = np.array([5.57, 4.67, 4.08, 3.84, 3.78])
    k_values = np.array([0.387, 0.140, 0.030, 0.015, 0.010])
    return (
        np.interp(wavelength_um, knots, n_values),
        np.interp(wavelength_um, knots, k_values),
    )


def materials_at(wavelength):
    n_si, k_si = silicon_nk(wavelength)
    return {
        "air": bz.Material(permittivity=1.0),
        "silica": bz.Material(permittivity=1.46**2),
        "sin": bz.Material(permittivity=n_sin**2),
        "silicon": material_from_nk(n_si, k_si, wavelength),
        "metal": bz.Material(permittivity=1.0, conductivity=3.5e7),
        **{
            channel: material_from_nk(*filter_nk(wavelength, channel), wavelength)
            for channel in ("red", "green", "blue")
        },
    }


In [ ]:
material_plot_wavelengths = np.linspace(0.40, 0.70, 301) * um
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
for channel, color in (("red", "r"), ("green", "g"), ("blue", "b")):
    nk = np.array([filter_nk(wavelength, channel) for wavelength in material_plot_wavelengths])
    axes[0].plot(material_plot_wavelengths / um, nk[:, 0], color=color, label=channel)
    axes[1].plot(material_plot_wavelengths / um, nk[:, 1], color=color, label=channel)
axes[0].set(ylabel="refractive index n", xlabel="wavelength (µm)")
axes[1].set(ylabel="extinction coefficient k", xlabel="wavelength (µm)")
axes[1].legend()
plt.show()


## 3. Build one wavelength-specific simulation

BeamZ structures are added in painter's order, so the detector is placed first and the overlying stack follows from bottom to top. The shield opening is a silica cylinder that overwrites the metal sheet. Four finite flux apertures cover 70% of each pixel width, matching the reference monitor fill factor.

A wide, normally incident Gaussian beam spanning the periodic cell is used as the closest native BeamZ source to a uniform plane wave. The reported efficiency uses BeamZ's internally calibrated launched power.

In [ ]:
pixel_signs = {
    "blue": (+1, +1),
    "red": (-1, -1),
    "green1": (-1, +1),
    "green2": (+1, -1),
}


def box_from_bounds(lower, upper, material):
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    return bz.Box(
        center=tuple(0.5 * (lower + upper)),
        size=tuple(upper - lower),
        material=material,
    )


def make_sensor_design(wavelength):
    mat = materials_at(wavelength)
    # The reference geometry is centered at zero; Simulation supplies its size.
    design = bz.Design(background=mat["air"])
    z_bottom = -Lz / 2
    z_silicon_top = z_bottom + t_silicon
    z_silica_top = Lz / 2 - d_absorber_space - lens_height
    z_filter_top = z_silica_top - d_lens_to_filter
    z_filter_bottom = z_filter_top - t_filter
    z_shield = z_filter_bottom - d_filter_to_shield - t_shield / 2
    z_connect0 = z_shield - t_shield / 2 - d_shield_to_connect - t_connect / 2
    z_connect1 = z_connect0 - connect_gap - t_connect
    z_ar = z_connect1 - t_connect / 2 - d_connect_to_ar - t_ar / 2

    # Full spheres come first; later layers clip them into spherical caps.
    lens_center_z = Lz / 2 - d_absorber_space - lens_radius
    for sign_x, sign_y in pixel_signs.values():
        design += bz.Sphere(
            position=(sign_x * pixel_width / 2, sign_y * pixel_width / 2, lens_center_z),
            radius=lens_radius,
            material=mat["silica"],
        )

    # The silica host clips each sphere at the cap base and fills the stack.
    design += box_from_bounds(
        (-Lx / 2, -Ly / 2, z_silicon_top),
        (+Lx / 2, +Ly / 2, z_silica_top),
        mat["silica"],
    )

    # Silicon photodetector and quarter-wave antireflection layer.
    design += box_from_bounds(
        (-Lx / 2, -Ly / 2, z_bottom),
        (+Lx / 2, +Ly / 2, z_silicon_top),
        mat["silicon"],
    )
    design += bz.Box(center=(0, 0, z_ar), size=(Lx, Ly, t_ar), material=mat["sin"])

    # Two interconnect planes and their central ground bars.
    for center_x in (-connect_distance / 2, +connect_distance / 2):
        design += bz.Box(
            center=(center_x, 0, z_connect1),
            size=(connect_width, Ly, t_connect),
            material=mat["metal"],
        )
    for center_y in (-connect_distance / 2, +connect_distance / 2):
        design += bz.Box(
            center=(0, center_y, z_connect0),
            size=(Lx, connect_width, t_connect),
            material=mat["metal"],
        )
    design += bz.Box(center=(0, 0, z_connect1), size=(2 * connect_width, Ly, t_connect), material=mat["metal"])
    design += bz.Box(center=(0, 0, z_connect0), size=(Lx, 2 * connect_width, t_connect), material=mat["metal"])

    # Metal shield, then its circular silica opening.
    design += bz.Box(center=(0, 0, z_shield), size=(Lx, Ly, t_shield), material=mat["metal"])
    design += bz.Circle(
        position=(0, 0, z_shield - t_shield / 2),
        radius=shield_hole_radius,
        depth=t_shield,
        points=96,
        material=mat["silica"],
    )

    # RGGB filter quadrants.
    for pixel, (sign_x, sign_y) in pixel_signs.items():
        channel = "green" if pixel.startswith("green") else pixel
        design += bz.Box(
            center=(sign_x * pixel_width / 2, sign_y * pixel_width / 2, 0.5 * (z_filter_bottom + z_filter_top)),
            size=(pixel_width, pixel_width, t_filter),
            material=mat[channel],
        )
    return design, z_silicon_top, z_ar


In [ ]:
def make_sim(wavelength):
    design, z_silicon_top, z_ar = make_sensor_design(wavelength)
    frequency = bz.LIGHT_SPEED / wavelength
    pulse = bz.GaussianPulse(freq0=frequency, fwidth=frequency / 10)
    source = bz.GaussianBeamSource(
        center=(0, 0, (Lz - d_absorber_space) / 2),
        size=(Lx, Ly, 0),
        source_time=pulse,
        direction="-z",
        angle_theta=0.0,
        pol_angle=0.0,
        waist_radius=10.0 * max(Lx, Ly),
        wavelength=wavelength,
        power=1.0,
    )

    monitors = [
        bz.FieldMonitor(
            center=(0, 0, z_silicon_top - 0.001 * um),
            size=(Lx, Ly, 0), freqs=[frequency], fields=("Ex", "Ey"),
            name="field_xy_silicon",
        ),
        bz.FieldMonitor(
            center=(+pixel_width / 2, 0, 0),
            size=(0, Ly, Lz), freqs=[frequency], fields=("Ex", "Ey", "Ez"),
            name="field_yz_BG",
        ),
        bz.FieldMonitor(
            center=(-pixel_width / 2, 0, 0),
            size=(0, Ly, Lz), freqs=[frequency], fields=("Ex", "Ey", "Ez"),
            name="field_yz_GR",
        ),
    ]
    for pixel, (sign_x, sign_y) in pixel_signs.items():
        monitors.append(
            bz.FluxMonitor(
                center=(sign_x * pixel_width / 2, sign_y * pixel_width / 2, z_silicon_top),
                size=(0.7 * pixel_width, 0.7 * pixel_width, 0),
                freqs=[frequency],
                name=f"flux_si_{pixel}",
            )
        )

    grid_spec = bz.GridSpec.uniform(0.20 * um if test_mode else 0.025 * um)
    run_time = (8 if test_mode else 40) / pulse.fwidth
    simulation = bz.Simulation(
        design=design,
        size=(Lx, Ly, Lz),
        sources=[source],
        monitors=monitors,
        boundaries=[
            bz.Periodic(axes=("x", "y")),
            bz.Absorber(edges=("front", "back"), thickness=0.20 * um),
        ],
        grid_spec=grid_spec,
        run_time=run_time,
    )
    return simulation


## 4. Inspect the setup before running

The green-light model is representative of the common geometry. BeamZ setup plots operate directly on the vector geometry and do not run or compile the FDTD simulation. The first panel is the detector-plane `xy` section; the second is an `xz` section through the red/green side of the Bayer cell.

In [ ]:
sim_green = make_sim(lam_green)
print(f"grid = {sim_green.grid.shape} | time steps = {sim_green.num_steps}")
display(sim_green)

fig, axes = sim_green.plot(
    z=-Lz / 2 + t_silicon - 0.001 * um,
    y=-pixel_width / 2,
    figsize=(11, 4.5),
    source_markers=True,
    monitor_markers=True,
    show=False,
)
for axis in np.asarray(axes).flat:
    axis.grid(False)
fig.suptitle("RGGB sensor: detector plane and R–G1 stack section", y=1.02)
plt.show()


## 5. Run the wavelength sweep

Each point is an independent narrowband simulation because its material coefficients are evaluated at that wavelength. This cell is intentionally separate from setup and visualization so the model can be reviewed before any compute is started. The cell has not been executed in the checked-in notebook.

In [ ]:
wavelengths = (
    np.array([0.55]) * um
    if test_mode else np.linspace(0.40, 0.70, 31) * um
)
results_by_wavelength = {}
for wavelength in wavelengths:
    simulation = make_sim(wavelength)
    results_by_wavelength[float(wavelength / um)] = simulation.run(
        progress=not test_mode, backend=execution_backend
    )


## 6. Field profiles

As in the reference, inspect the two vertical cuts through blue/green and red/green pixels plus the detector plane. The full sweep includes exactly 450, 550, and 650 nm; reduced mode plots its one available wavelength.

In [ ]:
field_wavelengths_um = [0.55] if test_mode else [0.65, 0.55, 0.45]
for monitor_name, section_title in (
    ("field_yz_BG", "G1–B sensor cross-section"),
    ("field_yz_GR", "R–G2 sensor cross-section"),
):
    fig, axes = plt.subplots(1, len(field_wavelengths_um), figsize=(4 * len(field_wavelengths_um), 3.6), squeeze=False)
    for axis, wavelength_um in zip(axes.flat, field_wavelengths_um):
        result = results_by_wavelength[wavelength_um]
        result.plot_field(
            monitor_name, "Ex", frequency=bz.LIGHT_SPEED / (wavelength_um * um),
            val="abs^2", ax=axis, show=False,
        )
        axis.set_title(f"λ = {1000 * wavelength_um:.0f} nm")
    fig.suptitle(section_title)
    plt.tight_layout()
    plt.show()

fig, axes = plt.subplots(1, len(field_wavelengths_um), figsize=(4 * len(field_wavelengths_um), 3.6), squeeze=False)
for axis, wavelength_um in zip(axes.flat, field_wavelengths_um):
    result = results_by_wavelength[wavelength_um]
    result.plot_field(
        "field_xy_silicon", "Ex", frequency=bz.LIGHT_SPEED / (wavelength_um * um),
        val="abs^2", ax=axis, show=False,
    )
    axis.set_title(f"λ = {1000 * wavelength_um:.0f} nm")
fig.suptitle("Field intensity at the silicon detector")
plt.tight_layout()
plt.show()


## 7. Optical efficiency

Following the reference, define $OE=P_{CH}/P_0$. BeamZ's downward-facing detector flux is negative in the global +z convention, so it is negated and divided by the source's internally calibrated launched power. The two green pixels are summed into one green channel. The wide Gaussian illumination is the remaining source approximation when comparing absolute efficiency with the reference's uniform plane wave.

In [ ]:
efficiency = {name: [] for name in pixel_signs}
for wavelength in wavelengths:
    result = results_by_wavelength[float(wavelength / um)]
    incident_power = result.launched_power(source=0)
    for pixel in pixel_signs:
        detector_power = -float(np.asarray(result[f"flux_si_{pixel}"].flux).squeeze())
        efficiency[pixel].append(max(detector_power, 0.0) / incident_power)

efficiency = {name: np.asarray(values) for name, values in efficiency.items()}
efficiency_green = efficiency["green1"] + efficiency["green2"]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(wavelengths / um, efficiency["red"], "r-o", ms=3, label="red")
ax.plot(wavelengths / um, efficiency_green, "g-o", ms=3, label="green1 + green2")
ax.plot(wavelengths / um, efficiency["blue"], "b-o", ms=3, label="blue")
for wavelength, color in ((lam_blue, "b"), (lam_green, "g"), (lam_red, "r")):
    ax.axvline(wavelength / um, color=color, ls="--", alpha=0.5)
ax.set(xlabel="wavelength (µm)", ylabel="optical efficiency", xlim=(0.40, 0.70), ylim=(0, None))
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## Takeaways and next steps

This BeamZ version retains the reference design's physical stack, periodic RGGB layout, narrowband field inspection, and channel-efficiency analysis. The most consequential differences are explicit: a wide Gaussian beam instead of an infinite plane wave, wavelength-by-wavelength material evaluation instead of a single causal broadband fit, and illustrative material data rather than a foundry stack.

For quantitative sensor design, replace the illustrative $n,k$ functions with measured process data, perform spatial- and spectral-convergence studies, and revisit this model when BeamZ provides causal dispersive media and a native uniform plane-wave source.